In [0]:

df_selic = spark.table(
    "workspace.economic_data.silver_selic"
)

df_dolar = spark.table(
    "workspace.economic_data.silver_dolar"
)

display(df_selic.limit(5))
display(df_dolar.limit(5))


data,valor
2025-01-10,12.25
2025-01-11,12.25
2025-02-03,13.25
2025-02-08,13.25
2025-02-16,13.25


data,valor
2025-01-10,6.0965
2025-02-03,5.8687
2025-02-18,5.6979
2025-02-20,5.7019
2025-03-17,5.7082


In [0]:

from pyspark.sql.functions import col

df_indicadores = (
    df_selic.alias("s")
    .join(
        df_dolar.alias("d"),
        col("s.data") == col("d.data"),
        "full_outer"
    )
    .select(
        col("s.data").alias("data_selic"),
        col("s.valor").alias("selic_meta"),
        col("d.data").alias("data_dolar"),
        col("d.valor").alias("dolar_venda")
    )
)

display(df_indicadores.orderBy("data_selic").limit(20))


data_selic,selic_meta,data_dolar,dolar_venda
2025-01-01,12.25,null,null
2025-01-02,12.25,2025-01-02,6.2086
2025-01-03,12.25,2025-01-03,6.1563
2025-01-04,12.25,null,null
2025-01-05,12.25,null,null
2025-01-06,12.25,2025-01-06,6.1119
2025-01-07,12.25,2025-01-07,6.0741
2025-01-08,12.25,2025-01-08,6.1321
2025-01-09,12.25,2025-01-09,6.0896
2025-01-10,12.25,2025-01-10,6.0965


In [0]:

from pyspark.sql.functions import col, coalesce

df_indicadores = (
    df_selic.alias("s")
    .join(
        df_dolar.alias("d"),
        col("s.data") == col("d.data"),
        "full_outer"
    )
    .select(
        coalesce(
            col("s.data"),
            col("d.data")
        ).alias("data"),
        col("s.valor").alias("selic_meta"),
        col("d.valor").alias("dolar_venda")
    )
    .orderBy("data")
)

display(df_indicadores)


data,selic_meta,dolar_venda
2025-01-01,12.25,null
2025-01-02,12.25,6.2086
2025-01-03,12.25,6.1563
2025-01-04,12.25,null
2025-01-05,12.25,null
2025-01-06,12.25,6.1119
2025-01-07,12.25,6.0741
2025-01-08,12.25,6.1321
2025-01-09,12.25,6.0896
2025-01-10,12.25,6.0965


In [0]:

from pyspark.sql.functions import year, month

df_gold = (
    df_indicadores
    .withColumn("ano", year(col("data")))
    .withColumn("mes", month(col("data")))
)

display(df_gold.limit(15))


data,selic_meta,dolar_venda,ano,mes
2025-01-01,12.25,null,2025,1
2025-01-02,12.25,6.2086,2025,1
2025-01-03,12.25,6.1563,2025,1
2025-01-04,12.25,null,2025,1
2025-01-05,12.25,null,2025,1
2025-01-06,12.25,6.1119,2025,1
2025-01-07,12.25,6.0741,2025,1
2025-01-08,12.25,6.1321,2025,1
2025-01-09,12.25,6.0896,2025,1
2025-01-10,12.25,6.0965,2025,1


In [0]:

from pyspark.sql.functions import avg, round

df_gold_mensal = (
    df_gold
    .groupBy("ano", "mes")
    .agg(
        round(avg("selic_meta"), 2).alias("media_selic"),
        round(avg("dolar_venda"), 4).alias("media_dolar")
    )
    .orderBy("ano", "mes")
)

display(df_gold_mensal)


ano,mes,media_selic,media_dolar
2025,1,12.31,6.0218
2025,2,13.25,5.7656
2025,3,13.64,5.7468
2025,4,14.25,5.7837
2025,5,14.64,5.6674
2025,6,14.85,5.5471
2025,7,15.0,5.5285
2025,8,15.0,5.4469
2025,9,15.0,5.3674
2025,10,15.0,5.3855


In [0]:

(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.economic_data.gold_indicadores")
)

(
    df_gold_mensal.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.economic_data.gold_indicadores_mensais")
)

print("Tabelas Gold criadas com sucesso!")


Tabelas Gold criadas com sucesso!


In [0]:
%sql

SELECT *
FROM workspace.economic_data.gold_indicadores_mensais
ORDER BY ano, mes;


ano,mes,media_selic,media_dolar
2025,1,12.31,6.0218
2025,2,13.25,5.7656
2025,3,13.64,5.7468
2025,4,14.25,5.7837
2025,5,14.64,5.6674
2025,6,14.85,5.5471
2025,7,15.0,5.5285
2025,8,15.0,5.4469
2025,9,15.0,5.3674
2025,10,15.0,5.3855


In [0]:

from pyspark.sql.window import Window
from pyspark.sql.functions import last

janela = (
    Window
    .orderBy("data")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_gold = (
    df_indicadores
    .withColumn(
        "selic_meta",
        last("selic_meta", ignorenulls=True).over(janela)
    )
    .withColumn("ano", year(col("data")))
    .withColumn("mes", month(col("data")))
)

display(df_gold)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data,selic_meta,dolar_venda,ano,mes
2025-01-01,12.25,null,2025,1
2025-01-02,12.25,6.2086,2025,1
2025-01-03,12.25,6.1563,2025,1
2025-01-04,12.25,null,2025,1
2025-01-05,12.25,null,2025,1
2025-01-06,12.25,6.1119,2025,1
2025-01-07,12.25,6.0741,2025,1
2025-01-08,12.25,6.1321,2025,1
2025-01-09,12.25,6.0896,2025,1
2025-01-10,12.25,6.0965,2025,1


In [0]:

from pyspark.sql.functions import avg, round

df_gold_mensal = (
    df_gold
    .groupBy("ano", "mes")
    .agg(
        round(avg("selic_meta"), 2).alias("media_selic"),
        round(avg("dolar_venda"), 4).alias("media_dolar")
    )
    .orderBy("ano", "mes")
)

display(df_gold_mensal)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ano,mes,media_selic,media_dolar
2025,1,12.31,6.0218
2025,2,13.25,5.7656
2025,3,13.64,5.7468
2025,4,14.25,5.7837
2025,5,14.64,5.6674
2025,6,14.85,5.5471
2025,7,15.0,5.5285
2025,8,15.0,5.4469
2025,9,15.0,5.3674
2025,10,15.0,5.3855


In [0]:

(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.economic_data.gold_indicadores")
)

(
    df_gold_mensal.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.economic_data.gold_indicadores_mensais")
)

print("Tabelas Gold atualizadas com sucesso!")


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tabelas Gold atualizadas com sucesso!


In [0]:
%sql

SELECT *
FROM workspace.economic_data.gold_indicadores_mensais
ORDER BY ano, mes;


ano,mes,media_selic,media_dolar
2025,1,12.31,6.0218
2025,2,13.25,5.7656
2025,3,13.64,5.7468
2025,4,14.25,5.7837
2025,5,14.64,5.6674
2025,6,14.85,5.5471
2025,7,15.0,5.5285
2025,8,15.0,5.4469
2025,9,15.0,5.3674
2025,10,15.0,5.3855


In [0]:

from pyspark.sql.functions import avg, round

df_gold_mensal = (
    df_gold
    .groupBy("ano", "mes")
    .agg(
        round(avg("selic_meta"), 2).alias("media_selic"),
        round(avg("dolar_venda"), 4).alias("media_dolar")
    )
    .orderBy("ano", "mes")
)

display(df_gold_mensal)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ano,mes,media_selic,media_dolar
2025,1,12.31,6.0218
2025,2,13.25,5.7656
2025,3,13.64,5.7468
2025,4,14.25,5.7837
2025,5,14.64,5.6674
2025,6,14.85,5.5471
2025,7,15.0,5.5285
2025,8,15.0,5.4469
2025,9,15.0,5.3674
2025,10,15.0,5.3855


In [0]:

(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.economic_data.gold_indicadores")
)

(
    df_gold_mensal.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.economic_data.gold_indicadores_mensais")
)

print("Tabelas Gold atualizadas com sucesso!")


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tabelas Gold atualizadas com sucesso!


In [0]:
%sql

SELECT *
FROM workspace.economic_data.gold_indicadores_mensais
ORDER BY ano, mes;


ano,mes,media_selic,media_dolar
2025,1,12.31,6.0218
2025,2,13.25,5.7656
2025,3,13.64,5.7468
2025,4,14.25,5.7837
2025,5,14.64,5.6674
2025,6,14.85,5.5471
2025,7,15.0,5.5285
2025,8,15.0,5.4469
2025,9,15.0,5.3674
2025,10,15.0,5.3855


In [0]:

from pyspark.sql.functions import col, sum, when

df_gold.select(
    sum(when(col("data").isNull(), 1).otherwise(0)).alias("datas_nulas"),
    sum(when(col("selic_meta").isNull(), 1).otherwise(0)).alias("selic_nula"),
    sum(when(col("dolar_venda").isNull(), 1).otherwise(0)).alias("dolar_nulo")
).show()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+----------+----------+
|datas_nulas|selic_nula|dolar_nulo|
+-----------+----------+----------+
|          0|         0|       113|
+-----------+----------+----------+



In [0]:

df_gold.filter(
    col("dolar_venda").isNull()
).select(
    "data",
    "selic_meta",
    "dolar_venda"
).orderBy("data").show(20, truncate=False)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+----------+-----------+
|data      |selic_meta|dolar_venda|
+----------+----------+-----------+
|2025-01-01|12.25     |NULL       |
|2025-01-04|12.25     |NULL       |
|2025-01-05|12.25     |NULL       |
|2025-01-11|12.25     |NULL       |
|2025-01-12|12.25     |NULL       |
|2025-01-18|12.25     |NULL       |
|2025-01-19|12.25     |NULL       |
|2025-01-25|12.25     |NULL       |
|2025-01-26|12.25     |NULL       |
|2025-02-01|13.25     |NULL       |
|2025-02-02|13.25     |NULL       |
|2025-02-08|13.25     |NULL       |
|2025-02-09|13.25     |NULL       |
|2025-02-15|13.25     |NULL       |
|2025-02-16|13.25     |NULL       |
|2025-02-22|13.25     |NULL       |
|2025-02-23|13.25     |NULL       |
|2025-03-01|13.25     |NULL       |
|2025-03-02|13.25     |NULL       |
|2025-03-03|13.25     |NULL       |
+----------+----------+-----------+
only showing top 20 rows


In [0]:

from pyspark.sql.functions import dayofweek

df_gold.filter(
    col("dolar_venda").isNull()
).groupBy(
    dayofweek("data").alias("dia_semana")
).count().orderBy("dia_semana").show()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+-----+
|dia_semana|count|
+----------+-----+
|         1|   52|
|         2|    2|
|         3|    1|
|         4|    1|
|         5|    4|
|         6|    1|
|         7|   52|
+----------+-----+



In [0]:

from pyspark.sql.functions import count

df_gold.groupBy("data") \
    .agg(count("*").alias("quantidade")) \
    .filter(col("quantidade") > 1) \
    .show()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----+----------+
|data|quantidade|
+----+----------+
+----+----------+



In [0]:

df_gold.filter(
    (col("selic_meta") <= 0) |
    (col("dolar_venda") <= 0)
).select(
    "data",
    "selic_meta",
    "dolar_venda"
).show()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----+----------+-----------+
|data|selic_meta|dolar_venda|
+----+----------+-----------+
+----+----------+-----------+



In [0]:
df_gold.filter(
    col("dolar_venda").isNull()
).filter(
    dayofweek("data").between(2, 6)
).select(
    "data",
    "selic_meta",
    "dolar_venda"
).orderBy("data").show(20, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+----------+-----------+
|data      |selic_meta|dolar_venda|
+----------+----------+-----------+
|2025-01-01|12.25     |NULL       |
|2025-03-03|13.25     |NULL       |
|2025-03-04|13.25     |NULL       |
|2025-04-18|14.25     |NULL       |
|2025-04-21|14.25     |NULL       |
|2025-05-01|14.25     |NULL       |
|2025-06-19|15.0      |NULL       |
|2025-11-20|15.0      |NULL       |
|2025-12-25|15.0      |NULL       |
+----------+----------+-----------+



In [0]:
df_gold.printSchema()

root
 |-- data: date (nullable = true)
 |-- selic_meta: double (nullable = true)
 |-- dolar_venda: double (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)



In [0]:
from pyspark.sql.functions import year, month

df_gold.filter(
    (col("ano") != year(col("data"))) |
    (col("mes") != month(col("data")))
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----+----------+-----------+---+---+
|data|selic_meta|dolar_venda|ano|mes|
+----+----------+-----------+---+---+
+----+----------+-----------+---+---+



In [0]:
tabelas = [
    "bronze_selic",
    "silver_selic",
    "bronze_dolar",
    "silver_dolar",
    "gold_indicadores",
    "gold_indicadores_mensais"
]

for tabela in tabelas:
    quantidade = spark.table(
        f"workspace.economic_data.{tabela}"
    ).count()

    print(f"{tabela}: {quantidade} registros")

bronze_selic: 365 registros
silver_selic: 365 registros
bronze_dolar: 252 registros
silver_dolar: 252 registros
gold_indicadores: 365 registros
gold_indicadores_mensais: 12 registros


In [0]:
df_gold_mensal.select(
    "ano",
    "mes",
    "media_selic",
    "media_dolar"
).orderBy(
    "ano",
    "mes"
).show(15, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----+---+-----------+-----------+
|ano |mes|media_selic|media_dolar|
+----+---+-----------+-----------+
|2025|1  |12.31      |6.0218     |
|2025|2  |13.25      |5.7656     |
|2025|3  |13.64      |5.7468     |
|2025|4  |14.25      |5.7837     |
|2025|5  |14.64      |5.6674     |
|2025|6  |14.85      |5.5471     |
|2025|7  |15.0       |5.5285     |
|2025|8  |15.0       |5.4469     |
|2025|9  |15.0       |5.3674     |
|2025|10 |15.0       |5.3855     |
|2025|11 |15.0       |5.3409     |
|2025|12 |15.0       |5.4531     |
+----+---+-----------+-----------+



In [0]:
df_gold_mensal.groupBy(
    "ano",
    "mes"
).count().filter(
    col("count") > 1
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---+---+-----+
|ano|mes|count|
+---+---+-----+
+---+---+-----+



In [0]:
from pyspark.sql.functions import min, max, countDistinct

df_gold.select(
    countDistinct("data").alias("datas_distintas"),
    min("data").alias("primeira_data"),
    max("data").alias("ultima_data")
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------------+-------------+-----------+
|datas_distintas|primeira_data|ultima_data|
+---------------+-------------+-----------+
|            365|   2025-01-01| 2025-12-31|
+---------------+-------------+-----------+

